# SQL query from table names - Continued

In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY2')

## The old Prompt

In [3]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [4]:
context = [ {'role':'system', 'content':"""

#Create table 1:
CREATE TABLE employees (ID_usr int, name varchar(15));
#Table 1 insert:
INSERT INTO employees(ID_usr, name) VALUES (1, 'Jonas');
INSERT INTO employees(ID_usr, name) VALUES (2, 'Maria');

#Create table 2
CREATE TABLE salary(ID_usr int, year date, salary float32);
#Table 2 insert:
INSERT INTO salary(ID_usr, year, salary) VALUES (1, 1995, 5000);
INSERT INTO salary(ID_usr, year, salary) VALUES (2, 2000, 5300);

#Create table 3
CREATE TABLE studies(ID int, ID_usr int, educational_level int, Institution varchar(30), Years date, Speciality varcgar(90));
#Table 3 insert:
INSERT INTO studies(ID, ID_usr, educational_level, Institution, Years, Speciality) VALUES (1, 1, 2, 'UESB', 2009, 'Economy');
INSERT INTO studies(ID, ID_usr, educational_level, Institution, Years, Speciality) VALUES (2, 2, 1, 'USP', 2012, 'Economy');

EXAMPLES:
SELECT * FROM employees;
SELECT * FROM salary;
SELECT * FROM studies;

"""} ]



In [5]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
 -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.
WRITE IN YOUR CONTEXT QUERIES HERE
"""
})

In [6]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [8]:
#new
context_user = context.copy()
print(return_CCRMSQL("""tell me what is the employees count""", context_user))

```sql
SELECT COUNT(*) AS employees_count FROM employees;
```


In [9]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("tell me what is the employees count", old_context_user))

This is your SQL:
```sql
SELECT COUNT(*) AS employee_count FROM employees;
```

This SQL command will count the number of employees in the "employees" table and return the result as "employee_count".


In [10]:
#new
print(return_CCRMSQL("tell me the average salary per institution", context_user))

```sql
SELECT s.Institution, AVG(s.salary) AS avg_salary
FROM salary s
JOIN employees e ON s.ID_usr = e.ID_usr
JOIN studies st ON s.ID_usr = st.ID_usr
GROUP BY s.Institution;
```


In [11]:
#old
print(return_CCRMSQL("tell me the average salary per institution", old_context_user))

This is your SQL:
```sql
SELECT Institution, AVG(salary) AS average_salary
FROM employees
JOIN salary ON employees.ID_usr = salary.ID_usr
GROUP BY Institution;
```

This SQL query retrieves the average salary per institution by joining the "employees" and "salary" tables on the ID_usr column. It then calculates the average salary for each institution and displays the result with the corresponding institution name.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

In [12]:
#new
print(return_CCRMSQL("Give me the sql code for the best institution based on the highest salary", context_user))

```sql
SELECT s.ID_usr, e.name, st.Institution
FROM salary s
JOIN employees e ON s.ID_usr = e.ID_usr
JOIN studies st ON s.ID_usr = st.ID_usr
WHERE s.salary = (SELECT MAX(salary) FROM salary);
```


In [13]:
#old
print(return_CCRMSQL("Give me the sql code for the best institution based on the highest salary", old_context_user))

This is your SQL:
```sql
SELECT e.name, s.salary, st.Institution
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
JOIN studies st ON e.ID_usr = st.ID_usr
WHERE s.salary = (SELECT MAX(salary) FROM salary);
```

This SQL code retrieves the name of the employee, their salary, and the institution they attended that has the highest salary among all employees. It joins the three tables (employees, salary, and studies) based on the employee ID, then filters the result to only show the row with the highest salary.


In [14]:
#new
context_user = context.copy()
print(return_CCRMSQL("""How many educational levels exist?""", context_user))

```sql
SELECT COUNT(DISTINCT educational_level) AS num_educational_levels
FROM studies;
```


The first prompt I tested was using the New model, I asked for the best institution based on the highest salary alumni. It answered the question succesfully, however the final table had the columns `ID_usr`, `name` and `Institution`. It should only have returned `Institution`, but it gave a little flair returning the highest paid alumni. I would also have returned the highest salary on the table.
The second prompt I tested was the same prompt on the old model. It also answered correctly but returned with the columns `name`, `salary` and `Institution`. Overall this was a better selection than the new model.
The third prompt was spot on. no added fluff. It answered the question objectively and that was it.
It should be great to experiment more using the models differences. I expected the new model to win in every question, but this experiment has shown that this is not always true. Actually, the old model using the prompts I tried has a little bit advantage.
